In [ ]:
import torch
import torch.nn as nn
import numpy as np
import firebase_admin
from firebase_admin import credentials, db
from collections import deque
import time

# ==========================================
# 1. PYTORCH MODEL DEFINITION
# ==========================================
class LocomotionHybrid(nn.Module):
    def __init__(self, num_classes=7):
        super(LocomotionHybrid, self).__init__()
        
        self.conv_layer = nn.Sequential(
            # Block 1
            nn.Conv1d(6, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            # Block 2
            nn.Conv1d(64, 64, kernel_size=5, padding=2), 
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3)
        )
        
        # LSTM 
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=2, 
                            batch_first=True, bidirectional=True, dropout=0.4)
        
        self.fc = nn.Sequential(
            nn.Linear(128 * 2, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
       # x shape: [Batch, Seq_Len, 6]
        x = x.transpose(1, 2)  # [Batch, 6, Seq_Len]
        x = self.conv_layer(x)
        x = x.transpose(1, 2)  # [Batch, Reduced_Seq_Len, 64]
        
        x, _ = self.lstm(x)
        
        # Global Average Pooling across the time dimension
        x = torch.mean(x, dim=1) 
        return self.fc(x)

# ==========================================
# 2. LOAD TRAINED WEIGHTS
# ==========================================
# Initialize the model
model = LocomotionHybrid(num_classes=7)

# Load the weights (map to CPU so it runs anywhere)
try:
    model.load_state_dict(torch.load('/kaggle/input/notebooks/hasnaa1/customized-classification-of-locomotion-modes/best_model_finetuned.pth', map_location=torch.device('cpu')))
    model.eval()  # Set model to evaluation mode (turns off dropout)
    print("PyTorch Model loaded successfully.")
except Exception as e:
    print(f"Error loading model weights: {e}")
    exit()

# ==========================================
# 3. FIREBASE SETUP
# ==========================================
try:
    cred = credentials.Certificate("/kaggle/input/datasets/shiamaakamel/hasnaa-rehab/serviceAccountKey.json")
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://rehab-project-9b48d-default-rtdb.firebaseio.com'
    })
    print("Connected to Firebase.")
except Exception as e:
    print(f"Firebase Connection Error: {e}")
    exit()

# ==========================================
# 4. BUFFER & CLASS MAPPING
# ==========================================
# IMPORTANT: This must match the sequence length you used when training!
# I am setting it to 50 as a default. Change it to whatever you used.
WINDOW_SIZE = 50  
imu_buffer = deque(maxlen=WINDOW_SIZE)

# Update these to match your exact 7 classes!
CLASS_NAMES = {
    0: "Sitting", 
    1: "Standing", 
    2: "Walking", 
    3: "Running",
    4: "Stairs Up",
    5: "Stairs Down",
    6: "Falling"
}

# ==========================================
# 5. THE REAL-TIME INFERENCE LISTENER
# ==========================================
def on_sensor_update(event):
    data = event.data
    
    # Ensure we are looking at a valid IMU update, not a status update
    if isinstance(data, dict) and 'accel_x' in data:
        # 1. Extract the 6 features
        features = [
            float(data.get('accel_x', 0)),
            float(data.get('accel_y', 0)),
            float(data.get('accel_z', 0)),
            float(data.get('gyro_x', 0)),
            float(data.get('gyro_y', 0)),
            float(data.get('gyro_z', 0))
        ]
        
        # 2. Add to sliding window
        imu_buffer.append(features)
        
        # 3. Once buffer is full, run inference
        if len(imu_buffer) == WINDOW_SIZE:
            # Convert to numpy array, then to PyTorch Tensor
            input_array = np.array(imu_buffer)
            
            # Convert to tensor and add Batch dimension: [1, Seq_Len, 6]
            input_tensor = torch.tensor(input_array, dtype=torch.float32).unsqueeze(0)
            
            # 4. Predict (Disable gradients for faster inference)
            with torch.no_grad():
                output = model(input_tensor)
                
                # Get the index of the highest probability class
                _, predicted_idx = torch.max(output, 1)
                class_index = predicted_idx.item()
            
            # Map index to human-readable string
            locomotion_state = CLASS_NAMES.get(class_index, "Unknown")
            print(f"Predicted Class [{class_index}]: {locomotion_state}")
            
            # 5. Write Prediction to Firebase
            try:
                db.reference('/sensors/esp32_device/locomotion_status').set({
                    'class_id': class_index,
                    'state': locomotion_state,
                    'timestamp': int(time.time())
                })
            except Exception as e:
                print(f"Firebase Write Failed: {e}")

# ==========================================
# 6. START THE LISTENER
# ==========================================
print(f"🎧 Listening for IMU data... Waiting to collect {WINDOW_SIZE} samples.")
db.reference('/sensors/esp32_device').listen(on_sensor_update)

# Keep the script alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nSystem stopped.")

PyTorch Model loaded successfully.
Connected to Firebase.
🎧 Listening for IMU data... Waiting to collect 50 samples.


In [8]:
import pandas as pd
import firebase_admin
from firebase_admin import credentials, db
import time

# ==========================================
# 1. FIREBASE SETUP
# ==========================================
# Check if Firebase is already initialized to prevent Kaggle crash
if not firebase_admin._apps:
    cred = credentials.Certificate("/kaggle/input/datasets/shiamaakamel/hasnaa-rehab/serviceAccountKey.json")
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://rehab-project-9b48d-default-rtdb.firebaseio.com'
    })
    print("✅ Connected to Firebase.")
else:
    print("⚡ Firebase was already connected.")

ref = db.reference('/sensors/esp32_device')

# ==========================================
# 2. LOAD DATA FROM CSV
# ==========================================
csv_file_path = '/kaggle/input/datasets/hasnaa1/enabl3s-dataset/enabl3s_data/AB193/Raw/AB193_Circuit_001_raw.csv'
df = pd.read_csv(csv_file_path)

# Extract exactly 500 readings
readings_to_upload = df.head(500)

print(f"📊 Loaded {len(readings_to_upload)} rows. Starting upload simulation...")

# ==========================================
# 3. STREAM DATA TO FIREBASE (The Simulation)
# ==========================================
for index, row in readings_to_upload.iterrows():
    
    # 🎯 UPDATED WITH EXACT COLUMN NAMES
    sensor_data = {
        'accel_x': float(row['Right_Shank_Ax']), 
        'accel_y': float(row['Right_Shank_Ay']),
        'accel_z': float(row['Right_Shank_Az']),
        'gyro_x': float(row['Right_Shank_Gx']),
        'gyro_y': float(row['Right_Shank_Gy']),
        'gyro_z': float(row['Right_Shank_Gz'])
    }
    
    # Upload to Firebase (overwriting the node, just like the ESP32 does)
    ref.update(sensor_data)
    
    # Print progress every 50 rows
    if (index + 1) % 50 == 0:
        print(f"📤 Uploaded {index + 1}/500 readings...")
        
    # Pause briefly so Firebase has time to process the event 
    # and your PyTorch listener has time to catch it.
    time.sleep(0.02) 

print("🎉 Simulation complete! 500 readings uploaded.")

⚡ Firebase was already connected.
📊 Loaded 500 rows. Starting upload simulation...
📤 Uploaded 50/500 readings...
📤 Uploaded 100/500 readings...
📤 Uploaded 150/500 readings...
📤 Uploaded 200/500 readings...
📤 Uploaded 250/500 readings...
📤 Uploaded 300/500 readings...
📤 Uploaded 350/500 readings...
📤 Uploaded 400/500 readings...
📤 Uploaded 450/500 readings...
📤 Uploaded 500/500 readings...
🎉 Simulation complete! 500 readings uploaded.
